## RAG 

### Experto en responder preguntas para AgroTech

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [63]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [44]:
MODEL = "llama3"
DB_NAME = "test_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [45]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Configurar los dos objetos clave de LangChain: «retriever» y «llm»

#### Nota al margen sobre la «temperatura»:
- Controla el grado de diversidad de la salida
- Una temperatura de 0 significa que la salida debe ser predecible
- Una temperatura más alta proporciona mayor variedad en las respuestas

Hay quien describe la temperatura como algo parecido a la «creatividad», pero eso no es del todo correcto
- En realidad, controla qué tokens se seleccionan durante la inferencia
- «temperature=0» significa: seleccionar siempre el token con mayor probabilidad
- «temperature=1» suele significar: un token con un 10 % de probabilidad debería elegirse el 10 % de las veces

Nota: una temperatura de 0 no significa que los resultados vayan a ser siempre reproducibles. También es necesario establecer una semilla aleatoria. 

In [64]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(temperature=0, model="llama3")

### Estos objetos de LangChain implementan el método `invoke`

In [65]:
retriever.invoke("¿Quien es Arcadio?")

[Document(id='cc3ac312-32fa-4a2c-8631-7a409c204e7e', metadata={'source': '..\\knowledge-base\\empleados\\Alejandro Herrera.md', 'doc_type': 'empleados'}, page_content='# Alejandro Herrera\n\n## Resumen\n\n-   **Fecha de nacimiento:** 11 de noviembre de 1988\n-   **Puesto:** Responsable de Control de Calidad y Jefe de Empaquetado\n-   **Ubicación:** Gáldar, Gran Canaria (Islas Canarias)\n-   **Salario actual:** 29.500 €\n-   **Metadatos sugeridos para ChromaDB:** ` {"departamento": "operaciones_almacen", "especialidad": "calibrado_dop", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa'),
 Document(id='c5e6c5a4-a60c-4163-beaa-d934ef94bc9c', metadata={'doc_type': 'compañia', 'source': '..\\knowledge-base\\compañia\\resumen.md'}, page_content='## Desglose del Portafolio de Clientes\n\nLos 32 contratos activos de AgroLLM abarcan todo el espectro de la tecnología y la gestión agrícola:'),
 Document(id='9fd83b97-69f6-4c6d-9e67-ec8d8111a325', metadata={'source': '

In [66]:
llm.invoke("¿Quién es Arcadio?")

AIMessage(content='Un personaje interesante!\n\nArcadio puede referirse a varios personajes históricos y literarios. Aquí te presento algunos de ellos:\n\n1. **Arcadio (rey visigodo)**: Fue un rey visigodo que gobernó el Reino Visigodo en la península ibérica desde 584 hasta su muerte en 603. Es conocido por haber sido uno de los más importantes monarcas visigodos y por haber promovido la conversión al cristianismo.\n2. **Arcadio (personaje literario)**: Es un personaje principal en la novela "La Comedia Humana" del escritor francés Gustave Flaubert, publicada en 1869. Arcadio es el hijo de Madame Bovary y su marido, Charles Bovary. El personaje es conocido por ser un joven idealista y romántico que se siente atrapado en una sociedad burguesa.\n3. **Arcadio (mitología)**: En la mitología griega, Arcadio era el hijo de Zeus y la ninfa Calírroe. Fue rey de los arcadios, un pueblo que habitaba en la región de Arcadia, en Grecia.\n\nEn resumen, Arcadio puede referirse a un rey visigodo, un

In [54]:
SYSTEM_PROMPT_TEMPLATE = """
Eres un asistente experto y amable que representa a la empresa AgroTech.
Estás chateando con un usuario sobre AgroTech.
Utiliza el contexto proporcionado para responder a cualquier pregunta.
Si no sabes la respuesta, dilo.
Contexto:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    print(docs)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [68]:
answer_question("¿Quién es Arcadio?", [])

[Document(id='cc3ac312-32fa-4a2c-8631-7a409c204e7e', metadata={'doc_type': 'empleados', 'source': '..\\knowledge-base\\empleados\\Alejandro Herrera.md'}, page_content='# Alejandro Herrera\n\n## Resumen\n\n-   **Fecha de nacimiento:** 11 de noviembre de 1988\n-   **Puesto:** Responsable de Control de Calidad y Jefe de Empaquetado\n-   **Ubicación:** Gáldar, Gran Canaria (Islas Canarias)\n-   **Salario actual:** 29.500 €\n-   **Metadatos sugeridos para ChromaDB:** ` {"departamento": "operaciones_almacen", "especialidad": "calibrado_dop", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa'), Document(id='c5e6c5a4-a60c-4163-beaa-d934ef94bc9c', metadata={'doc_type': 'compañia', 'source': '..\\knowledge-base\\compañia\\resumen.md'}, page_content='## Desglose del Portafolio de Clientes\n\nLos 32 contratos activos de AgroLLM abarcan todo el espectro de la tecnología y la gestión agrícola:'), Document(id='9fd83b97-69f6-4c6d-9e67-ec8d8111a325', metadata={'source': '..

AIMessage(content='Lo siento, pero no hay información disponible sobre alguien llamado Arcadio en el contexto proporcionado. ¿Podrías proporcionar más detalles o contexto sobre quién es Arcadio y qué relación tiene con AgroTech? Estoy aquí para ayudarte.', additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-07-07T20:39:36.741926Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1266365900, 'load_duration': 176820400, 'prompt_eval_count': 494, 'prompt_eval_duration': 244937600, 'eval_count': 54, 'eval_duration': 800237100, 'logprobs': None, 'model_name': 'llama3', 'model_provider': 'ollama'}, id='lc_run--019f3e4e-bd31-7d81-b567-d5faaaff8a8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 494, 'output_tokens': 54, 'total_tokens': 548})

In [37]:
gr.ChatInterface(answer_question).launch()

c:\Users\Usuario\Desktop\AI-Engineering-Course\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!